# 01 — MEDCON Evaluation

**Objectif** : Évaluer la traduction GPT-4o sur les termes médicaux spécialisés
via MEDCON groupé, BLEU et COMET, puis corréler avec les annotations d'un médecin.

**Données utilisées**
- `data/corpus_WMT24.json` — 49 documents EN→FR (source + référence + GPT-4o)
- `data/cleaned_mesh_snomed_dico.json` — dictionnaire médical nettoyé (2 515 entrées)
- `data/dr_annotations.json` — annotations médecin (25 documents, score Likert 1–5)

> **Note** : Ce notebook utilise le dictionnaire **nettoyé**.
> La comparaison nettoyé vs non-nettoyé est dans `02_dict_comparison.ipynb`.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cloner le repo pour accéder à src/
!git clone https://github.com/YOUR_USERNAME/TER-medical-translation.git /content/TER
%cd /content/TER

In [ ]:
!pip install pyahocorasick sacrebleu unbabel-comet sentence-transformers --quiet

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from scipy.stats import pearsonr, spearmanr
import sacrebleu
import sys
sys.path.append('/content/TER')
from src.medcon import build_pairs, build_automaton, medcon_grouped

## Chemins

In [ ]:
DRIVE_ROOT    = '/content/drive/MyDrive/TER'
CORPUS_PATH   = f'{DRIVE_ROOT}/data/corpus_WMT24.json'
DICT_PATH     = f'{DRIVE_ROOT}/data/cleaned_mesh_snomed_dico.json'
ANN_PATH      = f'{DRIVE_ROOT}/data/dr_annotations.json'

---
## 1 — Chargement du dictionnaire et construction des automates

In [ ]:
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    raw_dict = json.load(f)

print(f"Dictionnaire : {len(raw_dict):,} entrées EN")
print("\n--- Aperçu des 5 premières entrées ---")
for en_term, fr_terms in list(raw_dict.items())[:5]:
    print(f"  {en_term!r:40s} -> {fr_terms}")

In [ ]:
pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print(f"Paires : {len(pairs):,}  |  Termes EN : {len(automaton_en):,}  |  Termes FR : {len(automaton_fr):,}")

# Test rapide
r_test = medcon_grouped(
    "The patient had a myocardial infarction and was treated for hypertension.",
    "Le patient a subi un infarctus du myocarde et a été traité pour hypertension artérielle.",
    pairs, automaton_en, automaton_fr
)
print(f"\nTest rapide -> P={r_test['precision']:.2f}  R={r_test['recall']:.2f}  F1={r_test['f1']:.2f}")

---
## 2 — MEDCON groupé sur GPT-4o

In [ ]:
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    test_docs = json.load(f)
print(f"Corpus : {len(test_docs)} documents")

In [ ]:
print("=" * 80)
print("MEDCON GROUPÉ — GPT-4o")
print("=" * 80)

results_gpt = []
for i, doc in enumerate(tqdm(test_docs, desc="MEDCON")):
    r = medcon_grouped(doc['text_en'], doc['gpt_translation'], pairs, automaton_en, automaton_fr)
    results_gpt.append({
        'doc_id'           : i,
        'source_en'        : doc['text_en'],
        'gpt_fr'           : doc['gpt_translation'],
        'ref_fr'           : doc.get('translation_fr', ''),
        'medcon_f1'        : r['f1'],
        'medcon_precision' : r['precision'],
        'medcon_recall'    : r['recall'],
        'n_expected'       : r['n_expected'],
        'n_found'          : r['n_found'],
        'n_match'          : r['n_match'],
        'en_concepts'      : r['en_concepts'],
        'matched'          : r['matched'],
        'missed'           : r['missed'],
        'extra'            : r['extra'],
    })

df_gpt = pd.DataFrame(results_gpt)

print(f"\n  Precision : {df_gpt['medcon_precision'].mean():.3f}")
print(f"  Recall    : {df_gpt['medcon_recall'].mean():.3f}")
print(f"  F1        : {df_gpt['medcon_f1'].mean():.3f}")
print(f"\n  Paires attendues (moy.) : {df_gpt['n_expected'].mean():.1f}")
print(f"  Paires trouvées  (moy.) : {df_gpt['n_found'].mean():.1f}")
print(f"  Paires matchées  (moy.) : {df_gpt['n_match'].mean():.1f}")

---
## 3 — BLEU et COMET sur GPT-4o

In [ ]:
print("Calcul BLEU...")
for i, doc in enumerate(tqdm(test_docs, desc="BLEU")):
    bleu = sacrebleu.sentence_bleu(doc['gpt_translation'], [doc['translation_fr']]).score
    df_gpt.loc[i, 'bleu'] = bleu
print(f"BLEU moyen : {df_gpt['bleu'].mean():.2f}")

In [ ]:
USE_COMET = False
try:
    from comet import download_model, load_from_checkpoint
    import torch

    print("Chargement COMET (wmt22-comet-da)...")
    comet_model  = load_from_checkpoint(download_model('Unbabel/wmt22-comet-da'))
    comet_data   = [{
        'src': doc['text_en'],
        'mt' : doc['gpt_translation'],
        'ref': doc['translation_fr']
    } for doc in test_docs]
    comet_output = comet_model.predict(
        comet_data, batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, score in enumerate(comet_output.scores):
        df_gpt.loc[i, 'comet'] = float(score)
    USE_COMET = True
    print(f"COMET moyen : {df_gpt['comet'].mean():.3f}")

except Exception as e:
    print(f"COMET indisponible : {e}")
    df_gpt['comet'] = np.nan

---
## 4 — Comparaison MEDCON / BLEU / COMET

In [ ]:
print("=" * 80)
print("COMPARAISON MEDCON vs BLEU vs COMET")
print("=" * 80)

print(f"\n{'Métrique':<14} {'Moyenne':>9} {'Std':>9} {'Min':>9} {'Max':>9}")
print("-" * 52)
for name, col in [('MEDCON F1', 'medcon_f1'), ('BLEU', 'bleu'), ('COMET', 'comet')]:
    if col in df_gpt and df_gpt[col].notna().any():
        s = df_gpt[col].dropna()
        print(f"{name:<14} {s.mean():>9.3f} {s.std():>9.3f} {s.min():>9.3f} {s.max():>9.3f}")

print("\n--- Corrélations entre métriques ---")
r_mb, _ = pearsonr(df_gpt['medcon_f1'], df_gpt['bleu'])
print(f"  MEDCON <-> BLEU  : r = {r_mb:+.3f}")
if USE_COMET:
    r_mc, _ = pearsonr(df_gpt['medcon_f1'], df_gpt['comet'])
    r_bc, _ = pearsonr(df_gpt['bleu'],      df_gpt['comet'])
    print(f"  MEDCON <-> COMET : r = {r_mc:+.3f}")
    print(f"  BLEU   <-> COMET : r = {r_bc:+.3f}")

---
## 5 — Analyse des erreurs de terminologie

In [ ]:
print("=" * 80)
print("ANALYSE DES ERREURS DE TERMINOLOGIE")
print("=" * 80 + "\n")

all_missed = []
for ml in df_gpt['missed']: all_missed.extend(ml)
missed_counter = Counter(all_missed)

print("TOP 30 PAIRES MANQUÉES (EN présent en source, FR absent en traduction) :")
for i, (lbl, count) in enumerate(missed_counter.most_common(30), 1):
    print(f"  {i:2d}. [{count:2d}x] {lbl}")

all_extra = []
for el in df_gpt['extra']: all_extra.extend(el)
extra_counter = Counter(all_extra)

print("\nTOP 30 PAIRES EXTRAS (FR dans traduction, EN absent de la source) :")
for i, (lbl, count) in enumerate(extra_counter.most_common(30), 1):
    print(f"  {i:2d}. [{count:2d}x] {lbl}")

print(f"\n  Paires manquées uniques : {len(missed_counter)}")
print(f"  Paires extras uniques   : {len(extra_counter)}")

---
## 6 — Exemples détaillés : meilleur / médian / pire document

In [ ]:
def afficher_doc(row, titre):
    print(f"\n{'#'*80}\n  {titre}  (doc #{row['doc_id']})\n{'#'*80}")
    print(f"\nSOURCE EN :\n{row['source_en'][:400]}...")
    print(f"\nTRADUCTION GPT-4o FR :\n{row['gpt_fr'][:400]}...")
    print(f"\n{'─'*60}")
    print(f"MEDCON  F1={row['medcon_f1']:.3f}  P={row['medcon_precision']:.3f}  R={row['medcon_recall']:.3f}")
    bleu_val = row.get('bleu', float('nan'))
    if not np.isnan(bleu_val):
        print(f"BLEU    {bleu_val:.1f}")
    print(f"Paires  attendues={row['n_expected']}  trouvées={row['n_found']}  matchées={row['n_match']}")
    print(f"\n[OK] MATCHÉES ({len(row['matched'])}) :")
    for c in row['matched'][:8]: print(f"   {c}")
    print(f"\n[X] MANQUÉES ({len(row['missed'])}) :")
    for c in row['missed'][:8]:  print(f"   {c}")
    print(f"\n[+] EXTRAS  ({len(row['extra'])}) :")
    for c in row['extra'][:8]:   print(f"   {c}")


best_id   = df_gpt['medcon_f1'].idxmax()
worst_id  = df_gpt['medcon_f1'].idxmin()
median_id = (df_gpt['medcon_f1'] - df_gpt['medcon_f1'].median()).abs().idxmin()

afficher_doc(df_gpt.iloc[best_id],   "MEILLEUR DOCUMENT (F1 max)")
afficher_doc(df_gpt.iloc[median_id], "DOCUMENT MÉDIAN")
afficher_doc(df_gpt.iloc[worst_id],  "PIRE DOCUMENT (F1 min)")

---
## 7 — Chargement des annotations médecin

In [ ]:
with open(ANN_PATH, 'r', encoding='utf-8') as f:
    annotations_raw = json.load(f)

medical_data = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    ann = doc['annotations'][0]
    score, errors = None, []
    for r in ann['result']:
        if r['from_name'] == 'translation_likert':
            score = int(r['value']['choices'][0])
        elif r['from_name'] == 'document_issue_types':
            errors = r['value']['choices']
    medical_data.append({
        'source_en'      : doc['data']['question'],
        'translation_fr' : doc['data']['translation'],
        'medical_score'  : score,
        'has_term_error' : 'Terminologie incohérente' in errors,
    })

print(f"Documents annotés   : {len(medical_data)}")
print(f"Score médecin moyen : {np.mean([d['medical_score'] for d in medical_data if d['medical_score']]):.2f}/5")
print(f"Avec erreur termo   : {sum(d['has_term_error'] for d in medical_data)} docs")

---
## 8 — MEDCON sur les annotations médecin

In [ ]:
for doc in tqdm(medical_data, desc="MEDCON"):
    r = medcon_grouped(doc['source_en'], doc['translation_fr'], pairs, automaton_en, automaton_fr)
    doc['medcon_f1']        = r['f1']
    doc['medcon_precision'] = r['precision']
    doc['medcon_recall']    = r['recall']

df_medical = pd.DataFrame(medical_data)
print(f"MEDCON F1 moyen : {df_medical['medcon_f1'].mean():.3f}")

---
## 9 — COMET-QE sur les annotations médecin

In [ ]:
USE_COMET_QE = False
try:
    from comet import download_model, load_from_checkpoint
    import torch

    for model_name in ['Unbabel/wmt20-comet-qe-da', 'Unbabel/eamt22-cometinho-da']:
        try:
            print(f"Tentative : {model_name}")
            comet_qe_model = load_from_checkpoint(download_model(model_name))
            print("OK")
            break
        except:
            continue

    comet_data   = [{'src': d['source_en'], 'mt': d['translation_fr']} for d in medical_data]
    comet_output = comet_qe_model.predict(
        comet_data, batch_size=4,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, doc in enumerate(medical_data):
        doc['comet_qe'] = float(comet_output.scores[i])
    df_medical   = pd.DataFrame(medical_data)
    USE_COMET_QE = True
    print(f"COMET-QE moyen : {df_medical['comet_qe'].mean():.3f}")

except Exception as e:
    print(f"COMET-QE indisponible : {e}")
    for doc in medical_data: doc['comet_qe'] = None
    df_medical = pd.DataFrame(medical_data)

---
## 10 — mBERT et Sentence-BERT sur les annotations médecin

In [ ]:
USE_MBERT = False
try:
    from transformers import BertTokenizer, BertModel
    import torch
    from sklearn.metrics.pairwise import cosine_similarity

    print("Chargement mBERT...")
    mbert_tok   = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    mbert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
    mbert_model.eval()

    def get_mbert_emb(text):
        inputs = mbert_tok(text, return_tensors='pt', max_length=512, truncation=True, padding=True)
        with torch.no_grad():
            out = mbert_model(**inputs)
        return out.last_hidden_state.mean(dim=1).squeeze().numpy()

    for doc in tqdm(medical_data, desc="mBERT"):
        sim = cosine_similarity([get_mbert_emb(doc['source_en'])], [get_mbert_emb(doc['translation_fr'])])[0][0]
        doc['mbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_MBERT  = True
    print(f"mBERT similarity moyen : {df_medical['mbert_sim'].mean():.3f}")

except Exception as e:
    print(f"mBERT indisponible : {e}")
    for doc in medical_data: doc['mbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

In [ ]:
USE_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print("Chargement Sentence-BERT...")
    sbert = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

    for doc in tqdm(medical_data, desc="Sentence-BERT"):
        sim = cosine_similarity([sbert.encode(doc['source_en'])], [sbert.encode(doc['translation_fr'])])[0][0]
        doc['sbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_SBERT  = True
    print(f"Sentence-BERT moyen : {df_medical['sbert_sim'].mean():.3f}")

except Exception as e:
    print(f"Sentence-BERT indisponible : {e}")
    for doc in medical_data: doc['sbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

---
## 11 — Corrélations avec le score médecin

In [ ]:
print("=" * 80)
print("CORRÉLATIONS AVEC LE SCORE MÉDECIN")
print("=" * 80 + "\n")

valid_mask = df_medical['medical_score'].notna()
y = df_medical.loc[valid_mask, 'medical_score']

metrics_corr = [('MEDCON F1', 'medcon_f1')]
if USE_COMET_QE : metrics_corr.append(('COMET-QE',      'comet_qe'))
if USE_MBERT    : metrics_corr.append(('mBERT',         'mbert_sim'))
if USE_SBERT    : metrics_corr.append(('Sentence-BERT', 'sbert_sim'))

print(f"{'Métrique':<18} {'Pearson r':>10} {'p-val':>8} {'Spearman ρ':>12} {'p-val':>8}")
print("─" * 62)

correlations = []
for name, col in metrics_corr:
    if col in df_medical.columns and df_medical.loc[valid_mask, col].notna().all():
        x = df_medical.loc[valid_mask, col]
        r_p, p_p = pearsonr(y, x)
        r_s, p_s = spearmanr(y, x)
        print(f"{name:<18} {r_p:>10.3f} {p_p:>8.4f} {r_s:>12.3f} {p_s:>8.4f}")
        correlations.append((name, col, r_p, p_p, r_s, p_s))

if correlations:
    best = max(correlations, key=lambda x: abs(x[2]))
    print(f"\n{'='*62}")
    print(f"Meilleure corrélation Pearson : {best[0]} (|r| = {abs(best[2]):.3f})")
    print(f"{'='*62}")

---
## 12 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MEDCON groupé — GPT-4o (cleaned dictionary)', fontsize=13, fontweight='bold')

# 1. Distribution F1
ax = axes[0]
ax.hist(df_gpt['medcon_f1'], bins=20, color='#3498db', edgecolor='black', alpha=0.85)
ax.axvline(df_gpt['medcon_f1'].mean(), color='red', linestyle='--', lw=2,
           label=f"Moy. {df_gpt['medcon_f1'].mean():.3f}")
ax.set_xlabel('MEDCON F1'); ax.set_ylabel('Documents')
ax.set_title('Distribution MEDCON F1')
ax.legend(); ax.grid(alpha=0.3)

# 2. Precision vs Recall
ax = axes[1]
ax.scatter(df_gpt['medcon_recall'], df_gpt['medcon_precision'],
           alpha=0.65, color='#e74c3c', edgecolors='white', s=60)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3)

# 3. Corrélations vs score médecin
ax = axes[2]
if correlations:
    names_c  = [c[0] for c in correlations]
    r_vals   = [c[2] for c in correlations]
    bar_cols = ['#27ae60' if r > 0 else '#e74c3c' for r in r_vals]
    bars = ax.barh(names_c, r_vals, color=bar_cols, edgecolor='black', linewidth=1.2)
    for bar, r in zip(bars, r_vals):
        x_pos = r + 0.015 if r > 0 else r - 0.015
        ax.text(x_pos, bar.get_y() + bar.get_height()/2,
                f'r={r:.3f}', va='center',
                ha='left' if r > 0 else 'right',
                fontweight='bold', fontsize=9)
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Pearson r')
    ax.set_title(f'Corrélations vs score médecin (n={valid_mask.sum()})')
    ax.set_xlim(-0.6, 0.6); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()